# GSM8K Pipeline — Version robuste (sans PEFT)
T5-small full fine-tune, 5 époques, GPU T4 (~40 min)

In [ ]:
!pip install -q torch transformers datasets accelerate sentencepiece

In [ ]:
import sys, re, json, math, time, torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq)
print('✓ Imports')

In [ ]:
# Codec ψ minimal
import numpy as np
PHI = (1+5**0.5)/2; HALF_PI = 3.14159/2; PI = 3.14159; ZERO = 0.0
CODE_MAP = {'ADD':3,'SUBTRACT':1,'MULTIPLY':2,'DIVIDE':5}

def encoder_ops(ops):
    frames = []; variables = {}; vc = 0
    for op in ops:
        on = op.get('op','').upper(); vc += 1; vn = f'e{vc}'
        if on == 'INIT':
            try: v = float(op.get('value',0))
            except: v = 0.0
            variables[vn] = v
            frames.append({'code':4,'amp':abs(v),'phase':0.0 if v>=0 else PI,'op':'INIT','var':vn,'value':v})
            continue
        if on == 'QUERY': continue
        # Dernière variable modifiée
        sv = list(variables.values())[-1] if variables else 0.0
        for k in ('value','multiplier','divisor'):
            raw = op.get(k)
            if isinstance(raw,(int,float)):
                opd = float(raw); break
        else: continue
        cd = CODE_MAP.get(on,3)
        if on == 'ADD': nv = sv + opd; ph = ZERO
        elif on == 'SUBTRACT': nv = sv - opd; ph = PI
        elif on == 'MULTIPLY': nv = sv * opd; ph = ZERO
        elif on == 'DIVIDE': nv = sv / opd if opd else sv; ph = -HALF_PI
        else: nv = sv * opd; ph = ZERO
        variables[vn] = nv
        frames.append({'code':cd,'amp':1.0,'phase':HALF_PI,'op':on,'var':vn,'value':None})
        d = abs(nv - sv)
        frames.append({'code':cd,'amp':d if d>1e-9 else 1.0,'phase':ph,'op':on,'var':vn,'value':nv})
    return frames

def decoder_trames(frames):
    z = 0.0+0.0j; fv = None
    for f in frames:
        z += f['amp']*np.exp(1j*f['phase'])
        if f.get('value') is not None: fv = f['value']
    return float(z.real) if fv is None else fv
print('✓ Codec ψ')

In [ ]:
# Parseur d'annotations
ANOT_RE = re.compile(r'<<([^>]+)>>')
OP_MAP = {'+':'ADD','-':'SUBTRACT','*':'MULTIPLY','/':'DIVIDE'}
def _nettoyer(e):
    e = e.replace('(','').replace(')','')
    e = re.sub(r'--','+',e); e = re.sub(r'\+-','-',e); e = re.sub(r'-\+','-',e)
    return e
def anot2ops(answer):
    ops = []; chain = None
    for m in ANOT_RE.finditer(answer):
        expr = m.group(1)
        if '=' not in expr: continue
        ce, rs = expr.split('=',1)
        try: result = float(rs)
        except: continue
        clean = _nettoyer(ce)
        if clean.startswith('+'): clean = clean[1:]; chain = chain or 0.0
        neg = 1.0
        if clean.startswith('-'): clean = clean[1:]; neg = -1.0
        tokens = re.findall(r'[+\-*/]|\d+(?:\.\d+)?', clean)
        if not tokens: continue
        try: cur = float(tokens[0])*neg
        except: continue
        if chain is None or abs(cur-chain)>1e-9:
            ops.append({'op':'INIT','value':cur}); chain = cur
        i = 1
        while i+1 <= len(tokens)-1:
            if i+1 >= len(tokens): break
            op = tokens[i]; ns = tokens[i+1]
            if op not in OP_MAP: break
            try: nxt = float(ns)
            except: break
            m = OP_MAP[op]
            if m=='ADD': ops.append({'op':'ADD','value':nxt}); cur += nxt
            elif m=='SUBTRACT': ops.append({'op':'SUBTRACT','value':nxt}); cur -= nxt
            elif m=='MULTIPLY': ops.append({'op':'MULTIPLY','multiplier':nxt}); cur *= nxt
            elif m=='DIVIDE': ops.append({'op':'DIVIDE','divisor':nxt}); cur = cur/nxt if nxt else cur
            i += 2
        chain = result
    return ops
def reponse_finale(answer):
    m = re.search(r'####\s*(-?\d+(?:\.\d+)?)', answer)
    return float(m.group(1)) if m else None
def ops2texte(ops):
    parts = []
    for o in ops:
        if o['op']=='INIT': parts.append(f"INIT({o['value']})")
        elif o['op']=='ADD': parts.append(f"ADD({o['value']})")
        elif o['op']=='SUBTRACT': parts.append(f"SUB({o['value']})")
        elif o['op']=='MULTIPLY': parts.append(f"MUL({o['multiplier']})")
        elif o['op']=='DIVIDE': parts.append(f"DIV({o['divisor']})")
    return ' '.join(parts)
print('✓ Parseur')

In [ ]:
print('📦 Construction dataset...')
train = load_dataset('gsm8k','main',split='train')
test = load_dataset('gsm8k','main',split='test')
exemples = []
for item in train:
    exp = reponse_finale(item['answer'])
    if exp is None: continue
    ops = anot2ops(item['answer'])
    if not ops: continue
    try:
        got = decoder_trames(encoder_ops(ops))
        if got is None or abs(got-exp)>1e-6: continue
    except: continue
    exemples.append({'input':item['question'],'target':ops2texte(ops)})
print(f'Dataset : {len(exemples)} paires')
# Gold score
ok_gold = 0
for item in test:
    exp = reponse_finale(item['answer'])
    if exp is None: continue
    ops = anot2ops(item['answer'])
    if not ops: continue
    try:
        got = decoder_trames(encoder_ops(ops))
        ok_gold += got is not None and abs(got-exp)<1e-6
    except: continue
print(f'Gold (codec+annotations) : {ok_gold}/{len(test)} ({100*ok_gold/len(test):.1f}%)')

In [ ]:
print('\n🚀 Fine-tuning T5-small complet...')
tok = AutoTokenizer.from_pretrained('google/flan-t5-small')
ds = Dataset.from_list(exemples)
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split['train'], split['test']
def tok_fn(b):
    inp = tok(['translate to operations: '+t for t in b['input']], max_length=384, truncation=True, padding=False)
    tgt = tok(b['target'], max_length=128, truncation=True, padding=False)
    inp['labels'] = tgt['input_ids']; return inp
cols = train_ds.column_names
train_t = train_ds.map(tok_fn, batched=True, remove_columns=cols)
val_t = val_ds.map(tok_fn, batched=True, remove_columns=cols)
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-small')
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
args = TrainingArguments(
    output_dir='/kaggle/working/t5_gsm8k',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=True,
    report_to='none',
    dataloader_num_workers=2,
    
)
trainer = Trainer(model=model, args=args, train_dataset=train_t, eval_dataset=val_t,
    data_collator=DataCollatorForSeq2Seq(tok, model=model, padding=True), processing_class=tok)
t0 = time.time()
trainer.train()
print(f'✓ Terminé en {(time.time()-t0)/60:.1f} min')
final_path = '/kaggle/working/t5_gsm8k_final'
model.save_pretrained(final_path)
tok.save_pretrained(final_path)
print(f'✓ Modèle sauvegardé : {final_path}')

In [ ]:
print('\n📊 Évaluation...')
model.eval()
if torch.cuda.is_available(): model = model.cuda()
def ops2seq(pred):
    OM = {'MUL':'MULTIPLY','SUB':'SUBTRACT','ADD':'ADD','DIV':'DIVIDE','INIT':'INIT'}
    ops = []
    for token in pred.replace('\n',' ').split():
        m = re.match(r'(INIT|MUL|SUB|ADD|DIV)\(([^)]+)\)', token.strip())
        if not m: continue
        op, v = m.group(1), m.group(2)
        try: v = float(v)
        except: continue
        mapped = OM.get(op)
        if not mapped: continue
        if mapped=='INIT': ops.append({'op':'INIT','value':v})
        elif mapped=='MULTIPLY': ops.append({'op':'MULTIPLY','multiplier':v})
        elif mapped=='DIVIDE': ops.append({'op':'DIVIDE','divisor':v})
        elif mapped=='SUBTRACT': ops.append({'op':'SUBTRACT','value':v})
        elif mapped=='ADD': ops.append({'op':'ADD','value':v})
    return ops
ok_model = 0
for item in test:
    exp = reponse_finale(item['answer'])
    if exp is None: continue
    inp = 'translate to operations: '+item['question']
    inputs = tok(inp, return_tensors='pt', max_length=384, truncation=True)
    if torch.cuda.is_available(): inputs = {k:v.cuda() for k,v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, num_beams=1)
    pred = tok.decode(out[0], skip_special_tokens=True)
    ops = ops2seq(pred)
    if not ops: continue
    try:
        got = decoder_trames(encoder_ops(ops))
        if got is not None and abs(got-exp)<1e-6: ok_model += 1
    except: continue
print(f'Score T5+codec : {ok_model}/{len(test)} ({100*ok_model/len(test):.1f}%)')
print(f'Score gold     : {ok_gold}/{len(test)} ({100*ok_gold/len(test):.1f}%)')
import json
with open('/kaggle/working/results.json','w') as f:
    json.dump({'gold':ok_gold/len(test),'model':ok_model/len(test)},f)
print('\n✅ Résultats dans /kaggle/working/results.json')
print('✅ Modèle dans /kaggle/working/t5_gsm8k_final/')